In [ ]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline

# ==========================================
# 1. LOAD DATA
# ==========================================
TRAIN_PATH = "/content/train.csv"
TEST_PATH = "/content/test.csv"
OUTPUT_PATH = "/content/predictions.csv"

print("Loading data...")
df_train = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)

target_col = "label"
id_col = "id"

sensor_cols = ["dc_voltage", "dc_current", "irradiance", "panel_temp"]

# Check expected columns
required_train = set(sensor_cols + [target_col])
required_test = set(sensor_cols + [id_col])

missing_train = required_train - set(df_train.columns)
missing_test = required_test - set(df_test.columns)

if missing_train:
    raise ValueError(f"Missing columns in train.csv: {sorted(missing_train)}")
if missing_test:
    raise ValueError(f"Missing columns in test.csv: {sorted(missing_test)}")

# Round the sensor values, as in the original approach
for col in sensor_cols:
    df_train[col] = pd.to_numeric(df_train[col], errors="coerce").round(4)
    df_test[col] = pd.to_numeric(df_test[col], errors="coerce").round(4)

y = df_train[target_col].astype(str)

# ==========================================
# 2. FEATURE ENGINEERING
# ==========================================
def make_features(df):
    """Build features from raw sensor columns."""
    X = df[sensor_cols].copy()

    # Core physical features
    X["total_power"] = X["dc_voltage"] * X["dc_current"]
    X["current_per_irradiance"] = (
        X["dc_current"] / X["irradiance"].clip(lower=1e-3)
    )
    X["voltage_per_temp"] = (
        X["dc_voltage"] / X["panel_temp"].clip(lower=1e-3)
    )

    # Additional ratios and interactions
    X["power_per_irradiance"] = (
        X["total_power"] / X["irradiance"].clip(lower=1e-3)
    )
    X["current_squared"] = X["dc_current"] ** 2
    X["voltage_squared"] = X["dc_voltage"] ** 2

    # Replace infinite values; the pipeline will impute any resulting missing values
    X = X.replace([np.inf, -np.inf], np.nan)
    return X


X_all = make_features(df_train)
X_test_all = make_features(df_test)

# Compare raw sensors alone with raw sensors plus engineered features
feature_sets = {
    "raw_sensors": sensor_cols,
    "raw_plus_engineered": list(X_all.columns),
}

# ==========================================
# 3. MODELS AND CROSS-VALIDATION
# ==========================================
# Macro F1 weights every class equally.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model_options = [
    (
        "RandomForest",
        RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=1,
            max_features="sqrt",
            class_weight=None,
            random_state=42,
            n_jobs=-1,
        ),
    ),
    (
        "RandomForest_balanced",
        RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=1,
            max_features="sqrt",
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
    ),
    (
        "ExtraTrees",
        ExtraTreesClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=1,
            max_features="sqrt",
            class_weight=None,
            random_state=42,
            n_jobs=-1,
        ),
    ),
    (
        "ExtraTrees_balanced",
        ExtraTreesClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=1,
            max_features="sqrt",
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
    ),
]

results = []
best_score = -1.0
best_name = None
best_pipeline = None
best_features = None
best_oof_predictions = None

print("\nEvaluating model and feature combinations...")

for feature_set_name, columns in feature_sets.items():
    X = X_all[columns]

    for model_name, model in model_options:
        pipeline = make_pipeline(
            SimpleImputer(strategy="median"),
            model,
        )

        # Out-of-fold predictions ensure each row is predicted by a model
        # that did not train on that row.
        oof_predictions = cross_val_predict(
            pipeline,
            X,
            y,
            cv=cv,
            method="predict",
            n_jobs=1,
        )

        score = f1_score(y, oof_predictions, average="macro")
        run_name = f"{model_name} | {feature_set_name}"
        results.append((run_name, score))

        print(f"{run_name}: Macro F1 = {score:.5f}")

        if score > best_score:
            best_score = score
            best_name = run_name
            best_pipeline = pipeline
            best_features = columns
            best_oof_predictions = oof_predictions

# ==========================================
# 4. SHOW BEST VALIDATION RESULT
# ==========================================
print("\n==========================================")
print(f"Best configuration: {best_name}")
print(f"Out-of-fold Macro F1: {best_score:.5f}")
print("==========================================\n")

print("Out-of-fold classification report:")
print(
    classification_report(
        y,
        best_oof_predictions,
        zero_division=0,
    )
)

# ==========================================
# 5. REFIT ON ALL TRAINING DATA
# ==========================================
print("\nFitting the best model on all labeled data...")
best_pipeline.fit(X_all[best_features], y)

# ==========================================
# 6. PREDICT TEST DATA AND SAVE
# ==========================================
print("Predicting test data...")
test_predictions = best_pipeline.predict(X_test_all[best_features])

submission = pd.DataFrame({
    id_col: df_test[id_col],
    target_col: test_predictions,
})

submission.to_csv(OUTPUT_PATH, index=False)
print(f"Saved predictions to: {OUTPUT_PATH}")
print(submission.head())

Loading data...

Evaluating model and feature combinations...
RandomForest | raw_sensors: Macro F1 = 0.98702
RandomForest_balanced | raw_sensors: Macro F1 = 0.98908
ExtraTrees | raw_sensors: Macro F1 = 0.99111
ExtraTrees_balanced | raw_sensors: Macro F1 = 0.99114
RandomForest | raw_plus_engineered: Macro F1 = 0.98951
RandomForest_balanced | raw_plus_engineered: Macro F1 = 0.99103
ExtraTrees | raw_plus_engineered: Macro F1 = 0.99186
ExtraTrees_balanced | raw_plus_engineered: Macro F1 = 0.99223

Best configuration: ExtraTrees_balanced | raw_plus_engineered
Out-of-fold Macro F1: 0.99223

Out-of-fold classification report:
               precision    recall  f1-score   support

  degradation       1.00      0.99      0.99       691
       normal       0.99      0.99      0.99     26587
 open_circuit       0.99      1.00      1.00       242
      shading       0.99      0.99      0.99     16251
short_circuit       1.00      0.98      0.99       246

     accuracy                           0

In [ ]:
# ==========================================
# 7. DAY-GROUPED VALIDATION + RESIDUAL FEATURES + SUBMISSION FILES
# ==========================================
SUBMISSION_DIR = "/content"        # on Kaggle use "/kaggle/working"
groups = df_train["day"]
labels_order = ["normal", "shading", "degradation", "short_circuit", "open_circuit"]

def fit_reference(df):
    """Normal-plant reference, fitted on NORMAL rows of the given training days only."""
    n = df[df[target_col] == "normal"]
    cI = np.linalg.lstsq(np.c_[np.ones(len(n)), n.irradiance], n.dc_current, rcond=None)[0]
    cV = np.linalg.lstsq(np.c_[np.ones(len(n)), n.panel_temp, np.log(n.irradiance)],
                         n.dc_voltage, rcond=None)[0]
    return cI, cV

def residual_features(df, ref):
    cI, cV = ref
    V, I, G, T = df.dc_voltage, df.dc_current, df.irradiance, df.panel_temp
    expV = cV[0] + cV[1] * T + cV[2] * np.log(G)
    f = pd.DataFrame(index=df.index)
    f["rI"] = I - (cI[0] + cI[1] * G)      # current residual
    f["rV"] = V - expV                      # voltage residual
    f["rVrel"] = f["rV"] / expV             # relative voltage residual
    f["I_G"] = I / G
    f["P_G"] = V * I / G
    f["V_I"] = V / (I + 0.1)
    f["G"] = G
    return f

BASE = ["rI", "rV", "rVrel", "I_G", "P_G", "V_I"]
variants = {"noG": BASE, "withG": BASE + ["G"]}

def make_model():
    return ExtraTreesClassifier(n_estimators=300, min_samples_leaf=3,
                                class_weight="balanced", n_jobs=-1, random_state=42)

test_preds = {}
for name, cols in variants.items():
    # leave-one-day-out CV; the reference is refitted inside every fold
    oof = np.empty(len(df_train), dtype=object)
    for d in sorted(groups.unique()):
        tr_idx, va_idx = np.where(groups != d)[0], np.where(groups == d)[0]
        trn, val = df_train.iloc[tr_idx], df_train.iloc[va_idx]
        ref = fit_reference(trn)
        model = make_model().fit(residual_features(trn, ref)[cols], y.iloc[tr_idx])
        oof[va_idx] = model.predict(residual_features(val, ref)[cols])

    print(f"\n===== {name}: leave-one-day-out macro F1 = {f1_score(y, oof, average='macro'):.4f}")
    print(classification_report(y, oof, labels=labels_order, digits=3, zero_division=0))
    deg_recall = {}
    for d in [8, 9, 11, 12]:
        m = ((groups == d) & (y == "degradation")).values
        deg_recall[d] = round((oof[m] == "degradation").mean(), 2)
    print("degradation recall by day:", deg_recall)

    # fit on ALL training days and predict test
    ref_all = fit_reference(df_train)
    final = make_model().fit(residual_features(df_train, ref_all)[cols], y)
    pred = final.predict(residual_features(df_test, ref_all)[cols])
    test_preds[name] = pred

    sub = pd.DataFrame({id_col: df_test[id_col], target_col: pred})
    assert len(sub) == len(df_test) and sub[id_col].is_unique
    assert set(sub[target_col]) <= set(labels_order)
    path = f"{SUBMISSION_DIR}/submission_resid_{name}.csv"
    sub.to_csv(path, index=False)
    print("saved", path)
    print(sub[target_col].value_counts().reindex(labels_order).to_string())

print("\ntest rows where noG and withG disagree:", int((test_preds["noG"] != test_preds["withG"]).sum()))


===== noG: leave-one-day-out macro F1 = 0.9078
               precision    recall  f1-score   support

       normal      0.962     0.939     0.950     26587
      shading      0.914     0.942     0.928     16251
  degradation      0.743     0.816     0.778       691
short_circuit      0.814     0.976     0.887       246
 open_circuit      0.992     1.000     0.996       242

     accuracy                          0.939     44017
    macro avg      0.885     0.935     0.908     44017
 weighted avg      0.940     0.939     0.939     44017

degradation recall by day: {8: np.float64(0.35), 9: np.float64(1.0), 11: np.float64(0.95), 12: np.float64(0.96)}
saved /content/submission_resid_noG.csv
label
normal           5703
shading          2835
degradation       354
short_circuit     365
open_circuit      360

===== withG: leave-one-day-out macro F1 = 0.9217
               precision    recall  f1-score   support

       normal      0.952     0.949     0.951     26587
      shading      0.926